In [0]:
bronze_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://bronze@stretailmartdev011.dfs.core.windows.net/website/customers/customers_website.csv")
)

display(bronze_df)

customer_id,first_name,last_name,email,phone,city,country,registration_date,marketing_opt_in
WC000001,Fay,Otto,thijn94@example.org,(024)-3254198,Achtmaal,Belgium,2024-06-01,Yes
WC000002,Koen,Meis,joeyhoogers@example.net,043-1402327,Neerijnen,Germany,2025-02-05,Yes
WC000003,Isa,Mansvelt,lois95@example.net,068 2615699,Sellingen,Belgium,2024-02-14,No
WC000004,Lisanne,Kolen,puk59@example.org,(028) 6192770,Molkwerum,France,2023-10-07,Yes
WC000005,Ali,Evers,jinthe91@example.com,+31(0)097 882653,Haghorst,Belgium,2025-04-24,Yes
WC000006,Mohamed,van der Maath,van-ghoerlemette@example.com,(035)-4074399,Stad aan 't Haringvliet,France,2024-06-11,No
WC000007,Lieke,van Leeuwen,billungzakaria@example.com,+31882 012303,Liempde,Belgium,2024-04-29,Yes
WC000008,Sienna,le Briel,daan64@example.org,0640 486746,Finsterwolde,Germany,2026-05-22,No
WC000009,Nathan,Kwaadland,kaylee82@example.com,+3123-9723486,Eagum,France,2025-07-22,No
WC000010,Marit,Verheij,pslagmolen@example.com,(076)-4005722,Loosdrecht,France,2024-04-22,Yes


In [0]:
print("Total Records:", bronze_df.count())

Total Records: 4020


In [0]:
from pyspark.sql.functions import col

null_customer = bronze_df.filter(
    col("customer_id").isNull()
)

display(null_customer)

customer_id,first_name,last_name,email,phone,city,country,registration_date,marketing_opt_in


In [0]:
from pyspark.sql.functions import count

duplicates = (
    bronze_df
    .groupBy("customer_id")
    .agg(count("*").alias("records"))
    .filter(col("records") > 1)
)

display(duplicates)

customer_id,records
WC003734,2
WC001417,2
WC000238,2
WC003188,2
WC002864,2
WC001190,2
WC002710,2
WC003857,2
WC001669,2
WC001051,2


In [0]:
invalid_email = bronze_df.filter(
    ~col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+$")
)

display(invalid_email)

customer_id,first_name,last_name,email,phone,city,country,registration_date,marketing_opt_in
WC000063,Thijn,Cadefau,@retailmart.com,+31(0)56 8917634,Moerstraten,Germany,2023-08-05,No
WC000232,Kevin,Bosman,customer@,0669 474117,Oudehaske,Belgium,2026-03-04,Yes
WC000402,Loes,van der Laarse,invalid.email,+31(0)65-0887717,Slenaken,Germany,2026-03-29,Yes
WC000907,Kyano,Maas,invalid.email,(067)-6782673,Berkenwoude,Germany,2025-09-12,Yes
WC000959,Elin,van Ham,customer@@example.com,+31(0)786-763741,Langelo,Belgium,2026-06-20,No
WC001018,Maria,der Kijnder,@retailmart.com,+31(0)561-943888,Frederiksoord,France,2023-10-01,Yes
WC001060,Senna,Stettyn,invalid.email,+31(0)158 501691,Oudorp,Belgium,2024-08-05,No
WC001291,Nisa,Le Grand,invalid.email,+31(0)06 7529539,Roosteren,Belgium,2025-07-27,No
WC001327,Josephine,Hekker,missing-at-symbol.com,+31864-916913,Bruinehaar,Germany,2026-04-16,Yes
WC001332,Sara,van 't Wel,customer@,+3196-4841106,Harmelen,Germany,2024-05-19,Yes


In [0]:
print("========== DATA QUALITY REPORT ==========")

print("Total Records :", bronze_df.count())

print("Null Customer IDs :", bronze_df.filter(col("customer_id").isNull()).count())

print("Duplicate Customers :",
      bronze_df.groupBy("customer_id")
      .count()
      .filter(col("count") > 1)
      .count())

print("Invalid Emails :", invalid_email.count())

print("========================================")

========== DATA QUALITY REPORT ==========
Total Records : 4020
Null Customer IDs : 0
Duplicate Customers : 20
Invalid Emails : 21


In [0]:
valid_customers = bronze_df.filter(
    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+$")
)

invalid_customers = bronze_df.filter(
    ~col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+$")
)

In [0]:
print("Valid Customers :", valid_customers.count())
print("Invalid Customers :", invalid_customers.count())

Valid Customers : 3959
Invalid Customers : 21


In [0]:
quarantine_path = (
    "abfss://quarantine@stretailmartdev011.dfs.core.windows.net/"
    "customers/website_invalid_emails"
)

(
    invalid_email.write
    .format("delta")
    .mode("overwrite")
    .save(quarantine_path)
)

print("Invalid customer records written to quarantine.")

Invalid customer records written to quarantine.


In [0]:
quarantine_df = (
    spark.read
    .format("delta")
    .load(quarantine_path)
)

display(quarantine_df)

customer_id,first_name,last_name,email,phone,city,country,registration_date,marketing_opt_in
WC000063,Thijn,Cadefau,@retailmart.com,+31(0)56 8917634,Moerstraten,Germany,2023-08-05,No
WC000232,Kevin,Bosman,customer@,0669 474117,Oudehaske,Belgium,2026-03-04,Yes
WC000402,Loes,van der Laarse,invalid.email,+31(0)65-0887717,Slenaken,Germany,2026-03-29,Yes
WC000907,Kyano,Maas,invalid.email,(067)-6782673,Berkenwoude,Germany,2025-09-12,Yes
WC000959,Elin,van Ham,customer@@example.com,+31(0)786-763741,Langelo,Belgium,2026-06-20,No
WC001018,Maria,der Kijnder,@retailmart.com,+31(0)561-943888,Frederiksoord,France,2023-10-01,Yes
WC001060,Senna,Stettyn,invalid.email,+31(0)158 501691,Oudorp,Belgium,2024-08-05,No
WC001291,Nisa,Le Grand,invalid.email,+31(0)06 7529539,Roosteren,Belgium,2025-07-27,No
WC001327,Josephine,Hekker,missing-at-symbol.com,+31864-916913,Bruinehaar,Germany,2026-04-16,Yes
WC001332,Sara,van 't Wel,customer@,+3196-4841106,Harmelen,Germany,2024-05-19,Yes


In [0]:
from pyspark.sql import functions as F

invalid_customers_with_reason = (
    invalid_email
    .withColumn("rejection_reason", F.lit("INVALID_EMAIL_FORMAT"))
    .withColumn("source_system", F.lit("Website"))
    .withColumn("quarantined_at", F.current_timestamp())
)

(
    invalid_customers_with_reason.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(quarantine_path)
)

print("Quarantine records updated with audit metadata.")

Quarantine records updated with audit metadata.


In [0]:
display(
    spark.read
    .format("delta")
    .load(quarantine_path)
)

customer_id,first_name,last_name,email,phone,city,country,registration_date,marketing_opt_in,rejection_reason,source_system,quarantined_at
WC000063,Thijn,Cadefau,@retailmart.com,+31(0)56 8917634,Moerstraten,Germany,2023-08-05,No,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC000232,Kevin,Bosman,customer@,0669 474117,Oudehaske,Belgium,2026-03-04,Yes,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC000402,Loes,van der Laarse,invalid.email,+31(0)65-0887717,Slenaken,Germany,2026-03-29,Yes,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC000907,Kyano,Maas,invalid.email,(067)-6782673,Berkenwoude,Germany,2025-09-12,Yes,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC000959,Elin,van Ham,customer@@example.com,+31(0)786-763741,Langelo,Belgium,2026-06-20,No,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC001018,Maria,der Kijnder,@retailmart.com,+31(0)561-943888,Frederiksoord,France,2023-10-01,Yes,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC001060,Senna,Stettyn,invalid.email,+31(0)158 501691,Oudorp,Belgium,2024-08-05,No,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC001291,Nisa,Le Grand,invalid.email,+31(0)06 7529539,Roosteren,Belgium,2025-07-27,No,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC001327,Josephine,Hekker,missing-at-symbol.com,+31864-916913,Bruinehaar,Germany,2026-04-16,Yes,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z
WC001332,Sara,van 't Wel,customer@,+3196-4841106,Harmelen,Germany,2024-05-19,Yes,INVALID_EMAIL_FORMAT,Website,2026-08-04T10:03:58.206765Z


In [0]:
total_count = bronze_df.count()
invalid_count = invalid_email.count()
valid_count = total_count - invalid_count

print(f"Total records   : {total_count}")
print(f"Valid records   : {valid_count}")
print(f"Invalid records : {invalid_count}")
print(f"Reconciled      : {valid_count + invalid_count == total_count}")

Total records   : 4020
Valid records   : 3999
Invalid records : 21
Reconciled      : True
